In [1]:
import sys
sys.path.append("..")

In [2]:
import numpy as np
import pandas as pd
import os
import plotly.express as px
from sklearn.model_selection import ParameterGrid

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

In [3]:
# Grid Search LInf

def getStats(x: np.ndarray, theta: np.ndarray, x0: np.ndarray, lamb):
    return np.log(1 + np.exp(-(x @ theta))) + \
        (lamb * (np.linalg.norm(x0 - x, ord=1)))

def calThetaAdv_linf(xP: np.ndarray, weights: np.ndarray, bias, alpha):
    # xP does not have bias
    weights_adv = weights - (alpha * np.sign(xP))

    for i in range(len(xP)):
        if np.sign(xP[i]) == 0:
            weights_adv[i] = weights_adv[i] - (alpha * np.sign(weights_adv[i]))
    bias_adv = bias - alpha

    return np.hstack((weights_adv, bias_adv))

def getAllPossibleX(x0:np.ndarray, length=6, step_size=0.1):
    d = [[(length - 1) * -abs(x), length * abs(x)] for x in x0]
    d[x0.size- 1][0] = 1
    d[x0.size- 1][1] = 1

    delta_x = [np.arange(d[i][0], d[i][1] + step_size/2, step_size) for i in range(len(x0))]
    X = np.array(np.meshgrid(*delta_x)).T.reshape(-1, x0.shape[0])
    return np.round(X, decimals=5)

In [172]:
file_path = "../results/cost_validity/lr_synthetic_alg1_lamb0.1.pickle"
ret = pd.read_pickle(file_path)

x0 = ret['x_0'][1][0]
x0_withBias = np.hstack((x0, np.array([1])))
xR_old = ret['x_r'][1][0]
xR_old_withBias = np.hstack((xR_old, np.array([1])))

theta0 = ret['theta_0'][0][0][0][0].numpy().astype(np.float64)
bias0 = ret['theta_0'][0][0][1].numpy().astype(np.float64) 
divider = np.linalg.norm(theta0)
theta0 = ret['theta_0'][0][0][0][0].numpy().astype(np.float64) / divider
bias0 = ret['theta_0'][0][0][1].numpy().astype(np.float64) /divider
theta0_withBias = np.hstack((theta0, bias0))

alpha = 0.02
lamb = 0.2

In [173]:
lInfR = LARRecourse(weights=theta0, bias=bias0, alpha=alpha , lamb=lamb)
xR_new = lInfR.get_recourse(x0)
xR_new_withBias = np.hstack((xR_new, np.array([1])))

In [174]:
X = getAllPossibleX(np.hstack((x0, np.array([1]))), length=3)

In [175]:
Js = np.apply_along_axis(lambda x : getStats(x, calThetaAdv_linf(x[:-1], theta0, bias0, alpha), x0_withBias, lamb), arr=X, axis=1)
Js_min_i = np.argmin(Js)
xR_new_GS = X[Js_min_i]

In [176]:
J_xR_old = getStats(xR_old_withBias, calThetaAdv_linf(xR_old, theta0, bias0, alpha), x0_withBias, lamb)
J_xR_new = getStats(xR_new_withBias, calThetaAdv_linf(xR_new, theta0, bias0, alpha), x0_withBias, lamb)
J_xR_new_GS = getStats(xR_new_GS, calThetaAdv_linf(xR_new_GS[:-1], theta0, bias0, alpha), x0_withBias, lamb)

print(f"X0 : {x0}")
print(f"Theta0 : {theta0_withBias}")
print(f"Alpha: {alpha}")
print(f"Lamb: {lamb}")
print(f"XR Old Linf: {xR_old}, J : {J_xR_old}")
print(f"XR New Linf: {xR_new}, J : {J_xR_new}")
print(f"XR GS: {xR_new_GS[:-1]}, J : {J_xR_new_GS}")

X0 : [1.8468858  5.56784421]
Theta0 : [ 0.98652224 -0.16362725 -0.92027926]
Alpha: 0.02
Lamb: 0.2
XR Old Linf: [3.46585983 5.56784421], J : 0.5467686061661317
XR New Linf: [3.42075268 5.56784421], J : 0.5466139733021314
XR GS: [3.40623 5.56431], J : 0.5472012695443318


In [12]:
### SBA

file_path = "../results/cost_validity/lr_sba_alg1_lamb0.1.pickle"
ret = pd.read_pickle(file_path)
print(ret['theta_0'])

x0 = ret['x_0'][0][0]
x0_withBias = np.hstack((x0, np.array([1])))
xR_old = ret['x_r'][0][0]
xR_old_withBias = np.hstack((xR_old, np.array([1])))

theta0 = ret['theta_0'][0][0][0].numpy().astype(np.float64)
bias0 = ret['theta_0'][0][1].numpy().astype(np.float64)
divider = np.linalg.norm(theta0, 2)
theta0 = ret['theta_0'][0][0][0].numpy().astype(np.float64) / divider
bias0 = ret['theta_0'][0][1].numpy().astype(np.float64) /divider
theta0_withBias = np.hstack((theta0, bias0))

alpha = 0
lamb = 0.1

[(tensor([[ 0.7782, -0.2728,  0.1178, -0.1222, -0.0577,  0.2124, -0.0637,  0.2935,
          0.2429, -0.5569, -0.3784,  0.6732,  0.4216,  0.0846]]), tensor([-0.1420])), (tensor([[ 0.7782, -0.2728,  0.1178, -0.1222, -0.0577,  0.2124, -0.0637,  0.2935,
          0.2429, -0.5569, -0.3784,  0.6732,  0.4216,  0.0846]]), tensor([-0.1420])), (tensor([[ 0.7782, -0.2728,  0.1178, -0.1222, -0.0577,  0.2124, -0.0637,  0.2935,
          0.2429, -0.5569, -0.3784,  0.6732,  0.4216,  0.0846]]), tensor([-0.1420])), (tensor([[ 0.7782, -0.2728,  0.1178, -0.1222, -0.0577,  0.2124, -0.0637,  0.2935,
          0.2429, -0.5569, -0.3784,  0.6732,  0.4216,  0.0846]]), tensor([-0.1420])), (tensor([[ 0.7782, -0.2728,  0.1178, -0.1222, -0.0577,  0.2124, -0.0637,  0.2935,
          0.2429, -0.5569, -0.3784,  0.6732,  0.4216,  0.0846]]), tensor([-0.1420])), (tensor([[ 0.7782, -0.2728,  0.1178, -0.1222, -0.0577,  0.2124, -0.0637,  0.2935,
          0.2429, -0.5569, -0.3784,  0.6732,  0.4216,  0.0846]]), tensor([-0.

In [34]:
lInfR = LARRecourse(weights=theta0, bias=bias0, alpha=alpha , lamb=lamb)
xR_new = lInfR.get_recourse(x0)
xR_new_withBias = np.hstack((xR_new, np.array([1])))

In [35]:
J_xR_old = getStats(xR_old_withBias, calThetaAdv_linf(xR_old, theta0, bias0, alpha), x0_withBias, lamb)
J_xR_new = getStats(xR_new_withBias, calThetaAdv_linf(xR_new, theta0, bias0, alpha), x0_withBias, lamb)

print(f"X0 : {x0}")
print(f"Theta0 : {theta0_withBias}")
print(f"Alpha: {alpha}")
print(f"Lamb: {lamb}")
print(f"XR Old Linf: {xR_old}, J : {J_xR_old}")
print(f"XR New Linf: {xR_new}, J : {J_xR_new}")

X0 : [-0.26558718  0.93029434 -0.09333178 -0.4268808   0.28636168  1.76401877
  1.62572474 -0.27119077 -0.30590044  0.03956462  1.20519906  0.
  1.          0.        ]
Theta0 : [ 0.55058442 -0.19296914  0.08334686 -0.08644987 -0.04084672  0.15024761
 -0.04503808  0.20765633  0.17181497 -0.39397272 -0.26767812  0.47629433
  0.29829554  0.05986706 -0.10046115]
Alpha: 0
Lamb: 0.1
XR Old Linf: [ 1.88980785  0.93029434 -0.09333178 -0.4268808   0.28636168  1.76401877
  1.62572474 -0.27119077 -0.30590044  0.03956462  1.20519906  0.
  1.          0.        ], J : 0.5801774283940229
XR New Linf: [ 3.13281022  0.93029434 -0.09333178 -0.4268808   0.28636168  1.76401877
  1.62572474 -0.27119077 -0.30590044  0.03956462  1.20519906  0.
  1.          0.        ], J : 0.5402745832381457


In [30]:
# Newly Generated Files

file_path = "../results/cost_validity/mlp_sba_lime_roar.pickle"
ret = pd.read_pickle(file_path)

x0 = ret['x_0'][0][0]
x0_withBias = np.hstack((x0, np.array([1])))
xR_old = ret['x_r'][0][0]
xR_old_withBias = np.hstack((xR_old, np.array([1])))

theta0 = ret['theta_0']['out.weight'][0].numpy().astype(np.float64)
bias0 = ret['theta_0']['out.bias'].numpy().astype(np.float64)
divider = np.linalg.norm(theta0, 2)
theta0 = theta0 / divider
bias0 = bias0 /divider
theta0_withBias = np.hstack((theta0, bias0))

alpha = 0
lamb = 0.1

In [31]:
lInfR = LARRecourse(weights=theta0, bias=bias0, alpha=alpha , lamb=lamb)
xR_new = lInfR.get_recourse(x0)
xR_new_withBias = np.hstack((xR_new, np.array([1])))

In [32]:
J_xR_old = getStats(xR_old_withBias, calThetaAdv_linf(xR_old, theta0, bias0, alpha), x0_withBias, lamb)
J_xR_new = getStats(xR_new_withBias, calThetaAdv_linf(xR_new, theta0, bias0, alpha), x0_withBias, lamb)

print(f"X0 : {x0}")
print(f"Theta0 : {theta0_withBias}")
print(f"Alpha: {alpha}")
print(f"Lamb: {lamb}")
print(f"XR Old Linf: {xR_old}, J : {J_xR_old}")
print(f"XR New Linf: {xR_new}, J : {J_xR_new}")

X0 : [-0.26558718  0.93029434 -0.09333178 -0.4268808   0.28636168  1.76401877
  1.62572474 -0.27119077 -0.30590044  0.03956462  1.20519906  0.
  1.          0.        ]
Theta0 : [ 0.55058438 -0.19296914  0.08334685 -0.08644987 -0.04084671  0.15024761
 -0.04503808  0.20765633  0.17181498 -0.39397272 -0.26767814  0.47629433
  0.29829558  0.0598671  -0.1004612 ]
Alpha: 0
Lamb: 0.1
XR Old Linf: [ 3.13281033  0.93029434 -0.09333178 -0.4268808   0.28636168  1.76401877
  1.62572474 -0.27119077 -0.30590044  0.03956462  1.20519906  0.
  1.          0.        ], J : 0.5402746115474555
XR New Linf: [ 3.13281033  0.93029434 -0.09333178 -0.4268808   0.28636168  1.76401877
  1.62572474 -0.27119077 -0.30590044  0.03956462  1.20519906  0.
  1.          0.        ], J : 0.5402746115474554
